# 0. load FGSM eps8 dataset / model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# GTSRB zip 파일 (drive)
base_dir = '/content/drive/MyDrive/26sp_ML/PR1'
zip_path = os.path.join(base_dir, 'GTSRB.zip')

# 압축 해제
clean_root = '/content/GTSRB'
clean_csv = os.path.join(clean_root, 'Test.csv')

if not os.path.exists(clean_csv):
    print('Extracting GTSRB.zip...')
    os.makedirs(clean_root, exist_ok=True)

    import zipfile
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(clean_root)

adv_root = os.path.join(base_dir, 'GTSRB_FGSM_eps8_255')

print('clean_csv:', clean_csv)
print('adv_root:', adv_root)
print('Test.csv exists:', os.path.exists(clean_csv))
print('FGSM eps8 folder exists:', os.path.exists(adv_root))

# Test.csv 로드
test_df = pd.read_csv(clean_csv)
print('test_df shape:', test_df.shape)
print(test_df.head())

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

base_dir = '/content/drive/MyDrive/26sp_ML/PR1'
model_path = os.path.join(base_dir, 'resnet18_gtsrb_best_full_model.pth')

model = torch.load(model_path, map_location=device, weights_only=False)
model = model.to(device)
model.eval()

print('model loaded from:', model_path)

# 2. Defense functions: median filter, FFT low-pass filter

## 1) median

In [ ]:
def apply_median_filter(x, kernel_size=7):
    # x: [B, C, H, W], values in [0, 1]
    pad = kernel_size // 2

    x_pad = F.pad(x, (pad, pad, pad, pad), mode='reflect')
    patches = x_pad.unfold(2, kernel_size, 1).unfold(3, kernel_size, 1)
    patches = patches.contiguous().view(*x.shape, -1)

    x_med = patches.median(dim=-1).values
    return torch.clamp(x_med, 0.0, 1.0)

# 2) Gaussian blur

In [ ]:
def apply_gaussian_blur(x, kernel_size=5, sigma=1.0):
    # x: [B, C, H, W], values in [0, 1]

    device = x.device
    channels = x.shape[1]
    pad = kernel_size // 2

    coords = torch.arange(kernel_size, device=device) - pad
    yy, xx = torch.meshgrid(coords, coords, indexing='ij')

    kernel = torch.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    kernel = kernel / kernel.sum()

    kernel = kernel.view(1, 1, kernel_size, kernel_size)
    kernel = kernel.repeat(channels, 1, 1, 1)

    x_pad = F.pad(x, (pad, pad, pad, pad), mode='reflect')
    x_blur = F.conv2d(x_pad, kernel, groups=channels)

    return torch.clamp(x_blur, 0.0, 1.0)

## 3) FFT low-pass

In [ ]:
def apply_fft_lowpass(x, keep_ratio=0.25):
    # x: [B, C, H, W], values in [0, 1]
    b, c, h, w = x.shape

    x_fft = torch.fft.fft2(x, dim=(-2, -1))
    x_fft = torch.fft.fftshift(x_fft, dim=(-2, -1))

    h_keep = int(h * keep_ratio)
    w_keep = int(w * keep_ratio)

    h1 = (h - h_keep) // 2
    h2 = h1 + h_keep
    w1 = (w - w_keep) // 2
    w2 = w1 + w_keep

    mask = torch.zeros_like(x_fft)
    mask[:, :, h1:h2, w1:w2] = 1

    x_filtered = x_fft * mask
    x_filtered = torch.fft.ifftshift(x_filtered, dim=(-2, -1))
    x_rec = torch.fft.ifft2(x_filtered, dim=(-2, -1)).real

    return torch.clamp(x_rec, 0.0, 1.0)

# 3. Filter result

## 0) Original adversarial image

In [ ]:
sample_idx = 0  # 바꿔가면서 확인

row = test_df.iloc[sample_idx]
rel_path = row['Path']
true_label = int(row['ClassId'])

image_path = os.path.join(adv_root, rel_path)
image_pil = Image.open(image_path).convert('RGB')

plt.figure(figsize=(4, 4))
plt.imshow(image_pil.resize((224, 224)))
plt.title(f'Original adversarial image\nindex={sample_idx} | true={true_label}')
plt.axis('off')
plt.show()

## 1) Median filter

In [ ]:
x = transforms.ToTensor()(image_pil).unsqueeze(0).to(device)   # [1, C, H, W]
x_median = apply_median_filter(x, kernel_size=7)

median_img = x_median.squeeze(0).detach().cpu().permute(1, 2, 0).numpy()

plt.figure(figsize=(4, 4))
plt.imshow(median_img)
plt.title(f'Median filtered image\nindex={sample_idx}')
plt.axis('off')
plt.show()

## 2) Gaussian blur

In [ ]:
x = transforms.ToTensor()(image_pil).unsqueeze(0).to(device)

x_gaussian = apply_gaussian_blur(x, kernel_size=5, sigma=1.0)

gaussian_img = x_gaussian.squeeze(0).detach().cpu().permute(1, 2, 0).numpy()

plt.figure(figsize=(4, 4))
plt.imshow(gaussian_img)
plt.title(f'Gaussian blurred image\nkernel=5, sigma=1.0 | index={sample_idx}')
plt.axis('off')
plt.show()

## 3) FFT low-pass

In [ ]:
x = transforms.ToTensor()(image_pil).unsqueeze(0).to(device)   # [1, C, H, W]

# FFT
x_fft = torch.fft.fft2(x, dim=(-2, -1))
x_fft_shift = torch.fft.fftshift(x_fft, dim=(-2, -1))

# keep_ratio  = 남기는 주파수 비율 (저주파)
keep_ratio = 0.25
B, C, h, w = x.shape
h_keep = max(1, int(round(h * keep_ratio)))
w_keep = max(1, int(round(w * keep_ratio)))

h1 = (h - h_keep) // 2
h2 = h1 + h_keep
w1 = (w - w_keep) // 2
w2 = w1 + w_keep

# 중앙 저주파만 남기는 mask
mask = torch.zeros_like(x_fft_shift)
mask[:, :, h1:h2, w1:w2] = 1

# 고주파 컷
x_fft_cut = x_fft_shift * mask

# 복원
x_ifft = torch.fft.ifftshift(x_fft_cut, dim=(-2, -1))
x_rec = torch.fft.ifft2(x_ifft, dim=(-2, -1)).real
x_rec = torch.clamp(x_rec, 0.0, 1.0)
rec_img = x_rec.squeeze(0).detach().cpu().permute(1, 2, 0).numpy()

# 시각화용 magnitude spectrum
spec_before = torch.log1p(torch.abs(x_fft_shift)).mean(dim=1).squeeze(0).detach().cpu().numpy()    # FFT 결과
spec_after = torch.log1p(torch.abs(x_fft_cut)).mean(dim=1).squeeze(0).detach().cpu().numpy()       # FFT cut 결과

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.imshow(spec_before, cmap='gray')
plt.title('FFT magnitude spectrum')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(spec_after, cmap='gray')
plt.title(f'After high-frequency cut\nkeep_ratio={keep_ratio}')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(rec_img)
plt.title('Reconstructed image')
plt.axis('off')

plt.tight_layout()
plt.show()

# 4. Accuracy after filter

In [ ]:
import os
import shutil
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

drive_adv_root = os.path.join(base_dir, 'GTSRB_FGSM_eps8_255')
adv_root = '/content/GTSRB_FGSM_eps8_255'

# 데이터 복사
if not os.path.exists(adv_root):
    print('Copying FGSM eps8 dataset to /content ...')
    shutil.copytree(drive_adv_root, adv_root)

#
class gtsrb_attack_dataset(Dataset):
    def __init__(self, df, root_dir):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.root_dir, row['Path'])
        label = int(row['ClassId'])

        image = Image.open(image_path).convert('RGB')
        image = self.transform(image)

        return image, label


mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def normalize_batch(x):
    return (x - mean) / std


attack_dataset = gtsrb_attack_dataset(test_df, adv_root)

attack_loader = DataLoader(
    attack_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


def evaluate_with_defense(loader, defense_fn=None, desc='Evaluating'):
    model.eval()
    correct = 0
    total = 0

    pbar = tqdm(loader, desc=desc, leave=True)

    with torch.no_grad():
        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if defense_fn is not None:
                images = defense_fn(images)

            images = normalize_batch(images)
            outputs = model(images)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix({'acc': f'{correct / total:.4f}', 'correct': correct, 'total': total})

    return correct / total, correct, total

In [ ]:
results = []

# Median filter
acc, correct, total = evaluate_with_defense(
    attack_loader,
    defense_fn=None,
    desc='FGSM 8/255 - no defense')
results.append(['FGSM 8/255', 'No defense', acc, correct, total])

# Median filter
acc, correct, total = evaluate_with_defense(
    attack_loader,
    defense_fn=lambda x: apply_median_filter(x, kernel_size=7),
    desc='FGSM 8/255 - median k=7')
results.append(['FGSM 8/255', 'Median filter', acc, correct, total])

# Gaussian filter
acc, correct, total = evaluate_with_defense(
    attack_loader,
    defense_fn=lambda x: apply_gaussian_blur(x, kernel_size=5, sigma=1.0),
    desc='FGSM 8/255 - gaussian blur k=5 sigma=1.0')
results.append(['FGSM 8/255', 'Gaussian blur', acc, correct, total])

# FFT lowpass filter
acc, correct, total = evaluate_with_defense(
    attack_loader,
    defense_fn=lambda x: apply_fft_lowpass(x, keep_ratio=0.25),
    desc='FGSM 8/255 - FFT low-pass keep=0.25')
results.append(['FGSM 8/255', 'FFT low-pass', acc, correct, total])

defense_result_df = pd.DataFrame(results, columns=['dataset', 'defense', 'accuracy', 'correct', 'total'])
defense_result_df

In [ ]:
df = defense_result_df.copy()
df['accuracy_percent'] = df['accuracy'] * 100

x = range(len(df))

plt.figure(figsize=(7, 5))
plt.plot(x, df['accuracy_percent'], marker='o')

plt.ylim(60, 80)
plt.ylabel('Accuracy (%)')
plt.xlabel('Defense method')
plt.title('FGSM 8/255 Defense Accuracy Comparison')

plt.xticks(x, df['defense'])

for i, acc in enumerate(df['accuracy_percent']):
    plt.text(i, acc + 0.5, f'{acc:.2f}%', ha='center')

plt.tight_layout()
plt.show()